# 04 – Modellvergleich (Leitfaden Punkt 8)

Vergleich der drei endgültigen Verfahren **unter denselben Bedingungen**: gleicher zeitlicher Split (368 Trainings-, 24 Testmonate), Schätzung nur auf den Trainingsdaten, Prognose aller 24 Testmonate auf einmal, gleiche Metriken aus `src/evaluation.py`.

| Verfahren | Festlegung (vor dem Test) |
|---|---|
| Seasonal Naive | Baseline: Wert desselben Monats im letzten Trainingsjahr |
| Holt-Winters | additive Saison, gedämpfter Trend; Wahl per AIC (`03_holtwinters.ipynb`) |
| SARIMA(1,0,1)(2,1,2)$_{12}$ mit Konstante | Wahl per AIC und Sparsamkeitsregel (`02_arima.ipynb`, Commit 2587682) |

Die Festlegungen werden hier nur übernommen, nicht verändert.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import load_umsatz
from src.evaluation import train_test_split
from src import vergleich
from src import holtwinters as hw
from src.plots import speichern

s = load_umsatz(str(ROOT / "data/raw/45212-0004_de.csv"))
train, test = train_test_split(s)
prognosen = vergleich.alle_prognosen(train, len(test))
print(f"Training: {train.index[0]:%m/%Y}–{train.index[-1]:%m/%Y} ({len(train)} Monate)")
print(f"Test:     {test.index[0]:%m/%Y}–{test.index[-1]:%m/%Y} ({len(test)} Monate)")

In [ ]:
tabelle = vergleich.vergleichstabelle(test, prognosen)
tabelle.round(3)

In [ ]:
sn, hwn, sar = list(prognosen)
print(f"Holt-Winters besser als SARIMA in {vergleich.paarvergleich(test, prognosen, hwn, sar)} von {len(test)} Monaten")
print(f"Holt-Winters besser als Seasonal Naive in {vergleich.paarvergleich(test, prognosen, hwn, sn)} von {len(test)} Monaten")
print(f"SARIMA besser als Seasonal Naive in {vergleich.paarvergleich(test, prognosen, sar, sn)} von {len(test)} Monaten")

In [ ]:
fig = vergleich.plot_vergleich(train, test, prognosen)
speichern(fig, "14_vergleich_prognosen");

In [ ]:
fehler = hw.fehler_je_monat(test, prognosen)
fig = hw.plot_fehler(fehler)
speichern(fig, "15_vergleich_fehler");

**Ergebnis:**
- **Holt-Winters (gedämpft)** ist auf allen drei Metriken am besten (RMSE 2,437; MAE 2,042; MAPE 2,146 %) und hat in 17 von 24 Testmonaten einen kleineren Fehler als die Baseline.
- **SARIMA(1,0,1)(2,1,2)$_{12}$** liegt mit RMSE 3,331 knapp hinter der Baseline (3,282) und ist nur in 10 von 24 Monaten besser als diese; Holt-Winters ist in 16 von 24 Monaten besser als SARIMA.
- **Systematischer Fehler:** SARIMA überschätzt deutlich (mittlerer Fehler +2,250), Holt-Winters leicht (+0,815), die Baseline unterschätzt (−0,942).
- **Verlauf:** Im zweiten Testjahr wird Holt-Winters besser (RMSE 2,694 → 2,150), SARIMA schlechter (3,091 → 3,554).
- **Antwort auf die Fragestellung:** Mit dem vorab gewählten ARIMA-Modell lässt sich der reale Umsatz auf diesem Testzeitraum nicht genauer prognostizieren als mit Holt-Winters; die Baseline wird nicht übertroffen.
- **Einschränkungen:** Bewertet wird nur ein Testfenster von 24 Monaten; der Unterschied zwischen SARIMA und Baseline ist klein. Nur zur Information erreicht das manuelle SARIMA-Modell K3 auf der Testmenge RMSE 2,41 (`02_arima.ipynb`, Abschnitt 7): Die Prognosegüte von SARIMA hängt hier stark von der Modellwahl ab.